# 🧠 Brain Tumor MRI Classification — Vision Transformers

This notebook implements **three ViT architectures** for classifying brain MRI scans:

| Architecture | Description | Best For |
|---|---|---|
| `vit_b16` | Pure ViT-Base/16 | Large datasets |
| `swin_base` | Swin Transformer Base | **Recommended — best accuracy** |
| `hybrid_vit` | ResNet-50 + Transformer | Small datasets |

### Setup
1. Runtime → **T4 GPU** (Runtime > Change runtime type)
2. Add your Kaggle credentials below
3. Choose your architecture in Cell 5
4. Run All

## 1. 🔧 Setup & GPU Check

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
!pip install -q kaggle seaborn scikit-learn

## 2. 📥 Download Dataset

In [ ]:
import os, json

KAGGLE_USERNAME = 'your_kaggle_username'  # ← CHANGE
KAGGLE_KEY      = 'your_kaggle_api_key'   # ← CHANGE

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/data --unzip
print('Download complete!')

In [ ]:
import shutil, random
from pathlib import Path
random.seed(42)

src_train = Path('/content/data/Training')
src_test  = Path('/content/data/Testing')
VAL_SPLIT = 0.2
CLASSES   = ['notumor', 'glioma', 'meningioma', 'pituitary']

for cls in CLASSES:
    imgs = sorted((src_train / cls).glob('*'))
    random.shuffle(imgs)
    n_val = int(len(imgs) * VAL_SPLIT)
    splits = {'val': imgs[:n_val], 'train': imgs[n_val:]}
    for split, files in splits.items():
        dst = Path(f'/content/dataset/{split}/{cls}')
        dst.mkdir(parents=True, exist_ok=True)
        for f in files: shutil.copy(f, dst / f.name)
    for f in (src_test / cls).glob('*'):
        dst = Path(f'/content/dataset/test/{cls}')
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dst / f.name)
    print(f'{cls}: train={len(imgs)-n_val}, val={n_val}')

## 3. 📊 Data Exploration

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import random

train_root = Path('/content/dataset/train')
classes = sorted([d.name for d in train_root.iterdir() if d.is_dir()])

fig, axes = plt.subplots(len(classes), 4, figsize=(14, 14))
fig.suptitle('Sample MRI Images — 4 Classes', fontsize=16, fontweight='bold')
for row, cls in enumerate(classes):
    imgs = random.sample(list((train_root / cls).glob('*')), 4)
    for col, img in enumerate(imgs):
        ax = axes[row][col]
        ax.imshow(mpimg.imread(str(img)), cmap='gray')
        ax.axis('off')
        if col == 0:
            ax.set_title(cls.replace('notumor','No Tumor').title(), fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. 🧱 Model Definitions — ViT, Swin, Hybrid

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as tv_models

# ── 1. Pure ViT-B/16 ──────────────────────────────────────────────────────
class ViTClassifier(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, dropout=0.2):
        super().__init__()
        weights = tv_models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = tv_models.vit_b_16(weights=weights)
        hidden_dim = self.backbone.hidden_dim  # 768
        self.backbone.heads = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 512),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)
    def freeze_backbone(self):
        for n, p in self.backbone.named_parameters():
            if 'heads' not in n: p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

# ── 2. Swin Transformer Base ───────────────────────────────────────────────
class SwinClassifier(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, dropout=0.2):
        super().__init__()
        weights = tv_models.Swin_B_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = tv_models.swin_b(weights=weights)
        in_features = self.backbone.head.in_features  # 1024
        self.backbone.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, 512),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)
    def freeze_backbone(self):
        for n, p in self.backbone.named_parameters():
            if 'head' not in n: p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

# ── 3. Hybrid CNN + ViT ───────────────────────────────────────────────────
class ConvPatchEmbed(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        r = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V1)
        self.cnn  = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool,
                                  r.layer1, r.layer2, r.layer3)
        self.proj = nn.Conv2d(1024, embed_dim, kernel_size=1)
    def forward(self, x):
        f = self.proj(self.cnn(x))
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), H, W

class HybridViTClassifier(nn.Module):
    def __init__(self, num_classes=4, embed_dim=768, depth=6, num_heads=8, dropout=0.2, img_size=224):
        super().__init__()
        self.patch_embed = ConvPatchEmbed(embed_dim)
        n_patches = (img_size // 16) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim*4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        B = x.shape[0]
        tokens, H, W = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos_embed
        tokens = self.norm(self.encoder(tokens))
        return self.head(tokens[:, 0])
    def freeze_cnn(self):
        for p in self.patch_embed.cnn.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.parameters(): p.requires_grad = True

print('Model classes defined.')

## 5. ⚙️ Choose Architecture & Build Model

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  Choose one:                                 ║
# ║    'vit_b16'    Pure ViT-B/16                ║
# ║    'swin_base'  Swin Transformer ← best      ║
# ║    'hybrid_vit' ResNet50 + Transformer        ║
# ╚══════════════════════════════════════════════╝
ARCHITECTURE = 'swin_base'

NUM_CLASSES  = 4
IMG_SIZE     = 224
BATCH_SIZE   = 16   # reduce to 8 if OOM on Swin/ViT
EPOCHS       = 30
WARMUP       = 5    # linear warmup epochs
FREEZE_EPOCHS = 5   # freeze backbone for first N epochs
LR           = 1e-4
USE_MIXUP    = True

if ARCHITECTURE == 'vit_b16':
    model = ViTClassifier(num_classes=NUM_CLASSES, pretrained=True)
elif ARCHITECTURE == 'swin_base':
    model = SwinClassifier(num_classes=NUM_CLASSES, pretrained=True)
else:
    model = HybridViTClassifier(num_classes=NUM_CLASSES, img_size=IMG_SIZE)

model = model.to(device)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Architecture : {ARCHITECTURE}')
print(f'Total params : {total:,}')
print(f'Trainable    : {trainable:,}')

## 6. 📦 DataLoaders + MixUp

In [ ]:
import numpy as np
from collections import Counter
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.08,0.08)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(p=0.15, scale=(0.02,0.08)),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        self.classes = sorted(d.name for d in self.root.iterdir() if d.is_dir())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples, self.targets = [], []
        for cls in self.classes:
            for p in sorted((self.root/cls).glob('*')):
                if p.suffix.lower() in {'.jpg','.jpeg','.png'}:
                    self.samples.append((str(p), self.class_to_idx[cls]))
                    self.targets.append(self.class_to_idx[cls])
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, label = self.samples[i]
        img = Image.open(p).convert('RGB')
        return (self.transform(img) if self.transform else img), label

class MixUpCollator:
    def __init__(self, alpha=0.4, num_classes=4):
        self.alpha = alpha; self.num_classes = num_classes
    def __call__(self, batch):
        imgs, labels = zip(*batch)
        imgs = torch.stack(imgs)
        labels = torch.tensor(labels)
        targets = torch.zeros(len(labels), self.num_classes)
        targets.scatter_(1, labels.unsqueeze(1), 1)
        lam = np.random.beta(self.alpha, self.alpha)
        idx = torch.randperm(imgs.size(0))
        return lam*imgs + (1-lam)*imgs[idx], lam*targets + (1-lam)*targets[idx]

train_ds = BrainTumorDataset('/content/dataset/train', train_transform)
val_ds   = BrainTumorDataset('/content/dataset/val',   val_transform)
test_ds  = BrainTumorDataset('/content/dataset/test',  val_transform)
CLASS_NAMES = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
    collate_fn=MixUpCollator(num_classes=NUM_CLASSES) if USE_MIXUP else None)
val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Classes: {CLASS_NAMES}')

## 7. 🏋️ Training with Warmup + Cosine LR

In [ ]:
import math, time
import torch.nn.functional as F

class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, opt, warmup_epochs, total_epochs, min_lr=1e-6, last_epoch=-1):
        self.warmup = warmup_epochs; self.total = total_epochs; self.min_lr = min_lr
        super().__init__(opt, last_epoch)
    def get_lr(self):
        e = self.last_epoch
        if e < self.warmup:
            scale = (e+1) / max(self.warmup, 1)
        else:
            p = (e-self.warmup) / max(self.total-self.warmup, 1)
            scale = self.min_lr + 0.5*(1-self.min_lr)*(1+math.cos(math.pi*p))
        return [base*scale for base in self.base_lrs]

# Layer-wise LR: higher LR for head, lower for backbone
head_params     = [p for n,p in model.named_parameters() if any(k in n for k in ['head','heads'])]
backbone_params = [p for n,p in model.named_parameters() if not any(k in n for k in ['head','heads'])]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR,    'weight_decay': 0.05},
    {'params': head_params,     'lr': LR*10, 'weight_decay': 0.0},
])
scheduler = WarmupCosineScheduler(optimizer, WARMUP, EPOCHS)

def soft_ce(logits, targets):
    log_p = F.log_softmax(logits, dim=-1)
    if targets.dim() == 1: return F.nll_loss(log_p, targets)
    return -(targets * log_p).sum(dim=-1).mean()

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            if training: optimizer.zero_grad()
            logits = model(images)
            loss   = soft_ce(logits, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(1)
            hard  = labels.argmax(1) if labels.dim()==2 else labels
            correct += (preds==hard).sum().item()
            total   += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(hard.cpu().numpy())
    return total_loss/total, 100.*correct/total, all_preds, all_labels

# Optional: freeze backbone initially
if FREEZE_EPOCHS > 0 and hasattr(model, 'freeze_backbone'):
    model.freeze_backbone()
    print(f'Backbone frozen for first {FREEZE_EPOCHS} epochs.')

history = {k:[] for k in ['train_loss','val_loss','train_acc','val_acc','lr']}
best_val_acc = 0.0

print(f'\nTraining {ARCHITECTURE} on {device} for {EPOCHS} epochs...')
print('='*75)

for epoch in range(1, EPOCHS+1):
    # Unfreeze after FREEZE_EPOCHS
    if epoch == FREEZE_EPOCHS+1 and hasattr(model,'unfreeze_backbone'):
        model.unfreeze_backbone()
        print(f'  → Backbone unfrozen at epoch {epoch}')

    t0 = time.time()
    tr_loss, tr_acc, _,_ = run_epoch(train_loader, training=True)
    vl_loss, vl_acc, preds, labels = run_epoch(val_loader, training=False)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    for k,v in zip(['train_loss','val_loss','train_acc','val_acc','lr'],
                   [tr_loss, vl_loss, tr_acc, vl_acc, lr]):
        history[k].append(v)

    saved = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                    'val_acc':vl_acc,'class_names':CLASS_NAMES,
                    'architecture':ARCHITECTURE}, '/content/best_model.pth')
        saved = '  ✓'

    print(f'Ep {epoch:03d}/{EPOCHS} | TrLoss {tr_loss:.4f} TrAcc {tr_acc:.1f}% | '
          f'VlLoss {vl_loss:.4f} VlAcc {vl_acc:.1f}% | LR {lr:.2e} | '
          f'{time.time()-t0:.0f}s{saved}')

print(f'\nBest Val Accuracy: {best_val_acc:.2f}%')

## 8. 📊 Training Curves

In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Training History — {ARCHITECTURE}', fontsize=14, fontweight='bold')

for ax, (tr,vl), title in zip(axes,
    [('train_loss','val_loss'),('train_acc','val_acc'),('lr','lr')],
    ['Loss','Accuracy (%)','Learning Rate']):
    if tr == 'lr':
        ax.plot(epochs, history['lr'], 'g-', lw=2)
    else:
        ax.plot(epochs, history[tr], 'b-o', ms=3, label='Train')
        ax.plot(epochs, history[vl], 'r-o', ms=3, label='Val')
        ax.legend()
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

## 9. 📈 Evaluation — Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Load best checkpoint
ckpt = torch.load('/content/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best model: epoch {ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.2f}%')

_, test_acc, test_preds, test_labels = run_epoch(test_loader, training=False)
print(f'Test Accuracy: {test_acc:.2f}%\n')
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(test_labels, test_preds)
fig, axes = plt.subplots(1, 2, figsize=(16,6))
sns.heatmap(cm, annot=True, fmt='d',   cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

## 10. 🔍 Attention Rollout Visualization

In [ ]:
import math
import torch.nn.functional as F

# --- Attention extractor via hooks ---
class AttentionExtractor:
    def __init__(self, model):
        self.attns = []
        self._handles = []
        for m in model.modules():
            if isinstance(m, torch.nn.MultiheadAttention):
                self._handles.append(m.register_forward_hook(self._hook))
    def _hook(self, module, inputs, output):
        if isinstance(output, tuple) and output[1] is not None:
            self.attns.append(output[1].detach().cpu())
    def clear(self): self.attns = []
    def remove(self):
        for h in self._handles: h.remove()

def rollout(attentions, discard=0.85):
    result = None
    for attn in attentions:
        if attn.dim() == 4: attn = attn[0].mean(0)
        elif attn.dim() == 3: attn = attn[0]
        n = attn.shape[-1]
        flat = attn.flatten()
        thr  = flat.kthvalue(int(discard*flat.numel())).values
        attn = torch.where(attn >= thr, attn, torch.zeros_like(attn))
        I    = torch.eye(n)
        attn = (attn + I) / 2
        attn = attn / attn.sum(-1, keepdim=True).clamp(1e-6)
        result = attn if result is None else attn @ result
    mask = result[0, 1:].numpy() if result is not None else np.ones(1)
    return (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)

# Pick one image per class
test_root = Path('/content/dataset/test')
model.eval()

fig, big_axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(13, 4*len(CLASS_NAMES)))
fig.suptitle(f'Attention Rollout — {ARCHITECTURE}', fontsize=15, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    img_files = list((test_root / cls).glob('*'))
    if not img_files:
        continue
    img_path = random.choice(img_files)
    raw      = Image.open(img_path).convert('RGB')
    tensor   = val_transform(raw).unsqueeze(0).to(device)

    extractor = AttentionExtractor(model)
    with torch.no_grad():
        logits = model(tensor)
    probs    = torch.softmax(logits, 1).squeeze().cpu().numpy()
    pred_cls = CLASS_NAMES[probs.argmax()]

    grid = IMG_SIZE // 16
    mask = rollout(extractor.attns)
    n    = min(len(mask), grid*grid)
    side = int(math.sqrt(n))
    mask2d = mask[:side*side].reshape(side, side)
    mask_up = F.interpolate(
        torch.tensor(mask2d).unsqueeze(0).unsqueeze(0).float(),
        size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False
    ).squeeze().numpy()
    extractor.remove()

    axes = big_axes[row]
    axes[0].imshow(raw.resize((IMG_SIZE, IMG_SIZE)), cmap='gray'); axes[0].axis('off')
    axes[0].set_title(f'True: {cls}', fontsize=10)
    axes[1].imshow(mask_up, cmap='jet'); axes[1].axis('off')
    axes[1].set_title('Attention Map', fontsize=10)
    axes[2].imshow(raw.resize((IMG_SIZE, IMG_SIZE)), cmap='gray')
    axes[2].imshow(mask_up, alpha=0.5, cmap='jet'); axes[2].axis('off')
    axes[2].set_title(f'Pred: {pred_cls} ({probs.max()*100:.1f}%)', fontsize=10,
                      color='green' if pred_cls==cls else 'red')

plt.tight_layout()
plt.savefig('/content/attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Attention maps saved → /content/attention_maps.png')

## 11. 💾 Download All Outputs

In [ ]:
from google.colab import files
for f in ['/content/best_model.pth',
          '/content/training_curves.png',
          '/content/confusion_matrix.png',
          '/content/attention_maps.png']:
    files.download(f)
    print(f'Downloaded: {f}')